Step 1: Install Required Dependencies


In [ ]:
! pip install --upgrade langchain openai langchain-openai chromadb  -U langchain-community tiktoken pymupdf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.5 MB/s eta 0:00:00

Step 2: Import Necessary Libraries and Configure the LLM

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain import LLMChain, PromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from google.colab import userdata
from langchain.llms import OpenAI
from pathlib import Path
import os
import json
import fitz

# Retrieve the OpenAI API key
openai_api_key = userdata.get("OPENAIKEY")

# Configure the OpenAI model
llmModel = ChatOpenAI(
    temperature=0,
    verbose=True,
    openai_api_key=openai_api_key,
    model="gpt-3.5-turbo"
)



Step 3: Load Domain Knowledge Documents

In [ ]:
domain_knowledge_folder = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Knowledge Base"
domain_pdfs = [os.path.join(domain_knowledge_folder, f) for f in os.listdir(domain_knowledge_folder) if f.endswith('.pdf')]

# Function to extract text from a PDF using PyMuPDF
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        doc = fitz.open(pdf_path)  # Open the PDF
        for page in doc:
            text += page.get_text()  # Extract text from each page
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
    return text

# Load all PDFs into a document collection
domain_docs = []
for pdf in domain_pdfs:
    text = extract_text_from_pdf(pdf)
    if text.strip():  # Only include documents with non-empty text
        domain_docs.append({"page_content": text, "metadata": {"source": pdf}})

print(f"Loaded {len(domain_docs)} documents into the knowledge base.")


Loaded 90 documents into the knowledge base.


Step 4: Create Vector Store for Domain Knowledge

In [ ]:
# Convert domain_docs to a list of Document objects
document_objects = [
    Document(page_content=doc["page_content"], metadata=doc["metadata"])
    for doc in domain_docs
]

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_documents = []
for doc in document_objects:
    chunks = text_splitter.split_text(doc.page_content)
    for chunk in chunks:
        split_documents.append(Document(page_content=chunk, metadata=doc.metadata))

# Initialize embeddings and vector store
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
vectorstore = Chroma.from_documents(split_documents, embeddings)
retriever = vectorstore.as_retriever()










Step 5: Load Assignment Document

In [ ]:
# Define the path to the assignment PDF
assignment_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Assignments/Submission 3 -TIMG-5103-Assignment1.pdf"

# Open the PDF and extract content
assignment_content = ""
try:
    with fitz.open(assignment_path) as pdf:
        for page in pdf:  # Iterate through each page
            assignment_content += page.get_text() + "\n"  # Extract text from the page
except Exception as e:
    print(f"Error processing the assignment PDF: {e}")

# Print a snippet of the content for debugging (optional)
print("Extracted Assignment Content:", assignment_content[:500])  # Preview the first 500 characters

Extracted Assignment Content:  
 
CASE STUDY OF A GENERATIVE  
AI APPLICATION  
Assignment #1  
Prompt Engineering in Business  
TIM 5103  
  
  
  

 
Kapa AI (https://www.kapa.ai/)  
Problem  
Developers and technical teams within organizations often face delays and inefficiencies 
when searching for answers to technical questions. Accessing relevant solutions through 
technical documentation can be a slow, time-consuming process, impacting productivity 
and problem-solving speed.  
Solution  
Kapa.ai addresses this proble


Step 6: Load Assignment Question

In [ ]:
# Define the path to the assignment question PDF
assignment_question_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Question.pdf"

# Open the PDF and extract content
assignment_question = ""
try:
    with fitz.open(assignment_question_path) as pdf:
        for page in pdf:  # Iterate through each page
            assignment_question += page.get_text() + "\n"  # Extract text from the page
except Exception as e:
    print(f"Error processing the assignment question PDF: {e}")

# Print a snippet of the content for debugging (optional)
print("Extracted Assignment Question:", assignment_question[:500])  # Preview the first 500 characters


Extracted Assignment Question: Evaluation
10
Case study of a Generative AI application (15%), Week 4
• Choose an existing, real solution that uses LLMs as a core component.
• Discuss how the selected tool could disrupt current solutions (5%).
• Analyze potential challenges for integrating this tool/model into the IT 
operation of an organization (5%).
• Describe potential cases of misuse of this tool/model in the organization's 
context (5%).
• Individual assignment.
• Due Date: Sunday, Oct 06, 11:59 PM 




Step 7: Load Rubric and Grading Prompt

In [ ]:
# Path to the rubric file
rubric_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Rubrics/assessment_rubric.txt"

# Load the prompt text
with open(rubric_path, "r") as file:
    rubric_and_prompt_text = file.read()

# Create the PromptTemplate
grading_prompt = PromptTemplate(
    input_variables=["assignment_question", "assignment_content", "rubric"],
    template=rubric_and_prompt_text
)

 Step 8: Create the Grading Chain

In [ ]:
grading_chain = LLMChain(
    llm=llmModel,
    prompt=grading_prompt
)

<ipython-input-8-98756a3f2e6a>:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  grading_chain = LLMChain(


Step 9: Set Up RetrievalQA for Enhanced Context

In [ ]:
grading_agent = RetrievalQA.from_chain_type(
    llm=llmModel,
    retriever=retriever,
    chain_type="stuff",
    verbose=True
)

Step 10: Run the Grading Process

In [ ]:
grading_feedback = grading_chain.run({
    "assignment_question": assignment_question,
    "assignment_content": assignment_content,
    "rubric": rubric_and_prompt_text
})

# Print grading feedback
print("Grading Feedback:\n", grading_feedback)

<ipython-input-10-87befa1d0414>:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  grading_feedback = grading_chain.run({


Grading Feedback:
 - Selected Solution: 
   - Score: 4
   - Feedback: The solution uses an LLM-based component, but its role within the business context lacks clarity. It would be beneficial to provide more specific examples or explanations to better align with the assignment objectives.

- Disruption Analysis:
   - Score: 3
   - Feedback: A disruption analysis is provided, but it does not fully align with disruption theory. Consider providing more concrete examples or explanations to strengthen the analysis.

- Integration Analysis:
   - Score: 2
   - Feedback: Only one integration challenge criterion is analyzed. It would be helpful to explore additional integration dimensions to provide a more comprehensive analysis.

- Misuse Risks Analysis:
   - Score: 4
   - Feedback: Two to four misuse risks are identified and analyzed. Consider identifying and analyzing more misuse risks to deepen the analysis further.

- Total Score: 13
- Overall Level: Level 3

Overall, the submission demonst

Step 11: Set Up the Reviewer Agent

In [ ]:
reviewer_prompt_path = "/content/drive/MyDrive/Colab Notebooks/Grading Agent Project/Rubrics/reviewer prompt 5.txt"
with open(reviewer_prompt_path, "r") as file:
    reviewer_prompt_text = file.read()

reviewer_prompt = PromptTemplate(
    input_variables=["grading_feedback", "rubric"],
    template=reviewer_prompt_text
)

reviewer_chain = LLMChain(
    llm=ChatOpenAI(model_name="gpt-4", temperature=0, openai_api_key=openai_api_key),
    prompt=reviewer_prompt,
)

Step 12: Run the Reviewer Agent Process

In [ ]:
reviewer_feedback = reviewer_chain.run({
    "feedback": grading_feedback,
    "rubric": rubric_and_prompt_text
})

# Print reviewer feedback
print("Reviewer Feedback:\n", reviewer_feedback)

# Enhanced Retrieval for Further Contextual Feedback
enhanced_feedback = grading_agent({"query": reviewer_feedback})
print("Enhanced Feedback with Context:\n", enhanced_feedback["result"])

Reviewer Feedback:
 Validation Against Rubric:

1. Selected Solution:
   - Score: 4
   - Feedback: The solution uses an LLM-based component, but its role within the business context lacks clarity. It would be beneficial to provide more specific examples or explanations to better align with the assignment objectives.
   - Validation: The score and feedback align with the rubric's Level 3 criteria for this section. No adjustments are needed.

2. Disruption Analysis:
   - Score: 3
   - Feedback: A disruption analysis is provided, but it does not fully align with disruption theory. Consider providing more concrete examples or explanations to strengthen the analysis.
   - Validation: The score and feedback align with the rubric's Level 2 criteria for this section. The score needs to be adjusted from 3 to 2.

3. Integration Analysis:
   - Score: 2
   - Feedback: Only one integration challenge criterion is analyzed. It would be helpful to explore additional integration dimensions to provide a

<ipython-input-12-6cfe18f580a9>:10: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  enhanced_feedback = grading_agent({"query": reviewer_feedback})



> Finished chain.
Enhanced Feedback with Context:
 Based on the provided information, the user's validation against the rubric and the refined feedback seems accurate and aligned with the criteria specified in the rubric. The adjustments made to the scores for the Disruption Analysis section were appropriate based on the feedback provided. The user has been given clear guidance on how to improve their analysis in each section to meet the rubric's criteria more effectively.
